In [1]:
# ============================================================
# Cell 1: 环境
# ============================================================
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.shared.source_artifact_service import create_source_artifact
from src.engineering_intelligence.extract_service import (
    extract_technical_candidates,
    persist_candidates,
    list_pending_candidates,
    approve_candidate,
    reject_candidate,
)

driver = get_driver()
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# ============================================================
# Cell 2: 准备一个 SourceArtifact
# ============================================================
src_id = create_source_artifact(
    driver,
    source_type="literature",
    title="Engineered phage with expanded host range against KL47 K. pneumoniae",
    url="https://example.com/paper-kl47-engineered-phage",
    published_date="2025-08-01",
    credibility_tier="secondary",
    actor_id="demo",
)
print(f"✅ SourceArtifact: {src_id}")

✅ SourceArtifact: SRC:7B7FE27F


In [3]:
# ============================================================
# Cell 3: 待抽取的原始文本（论文摘要，演示用）
# ============================================================
SAMPLE_TEXT = """
We report an engineered bacteriophage vB_Kpn_HRE_001 derived from the wild-type
phage PKP001, in which the tail fiber gene was replaced with a chimeric tail fiber
to expand host range. The engineered construct was tested in vitro against 12
clinical isolates of KL47-type Klebsiella pneumoniae. The engineered phage
demonstrated a 2.0-fold improvement in host coverage compared with the parental
phage, with a lysis efficiency of 98.5% in the tested strains. No cytotoxicity
was observed in human cell lines. However, the study is limited to in vitro
assays, and in vivo efficacy remains to be established. A patent application
covering the chimeric tail fiber sequence has been filed (WO2025/123456A1).
"""

print("文本长度：", len(SAMPLE_TEXT))

文本长度： 705


In [4]:
# ============================================================
# Cell 4: 运行 DeepSeek 抽取
# ============================================================
extraction = extract_technical_candidates(
    driver,
    source_artifact_id=src_id,
    source_text=SAMPLE_TEXT,
    actor_id="demo",
)
print("📊 抽取统计：")
print(json.dumps(extraction["stats"], ensure_ascii=False, indent=2))
print("\n📋 候选对象预览：")
print(json.dumps(extraction["candidates"], ensure_ascii=False, indent=2)[:3000])

📊 抽取统计：
{
  "strategies": 2,
  "constructs": 1,
  "claims": 4,
  "results": 3,
  "repaired": false,
  "parse_error": false
}

📋 候选对象预览：
{
  "candidate_strategies": [
    {
      "strategy_type": "tail_fiber_engineering",
      "description": "将野生型噬菌体 PKP001 的尾丝基因替换为嵌合尾丝，以扩展宿主范围。",
      "exact_quote": "in which the tail fiber gene was replaced with a chimeric tail fiber to expand host range",
      "char_offset": 108
    },
    {
      "strategy_type": "host_range_expansion",
      "description": "通过嵌合尾丝替换扩展工程噬菌体的宿主范围。",
      "exact_quote": "to expand host range",
      "char_offset": 168
    }
  ],
  "candidate_constructs": [
    {
      "public_name": "vB_Kpn_HRE_001",
      "construct_code": "vB_Kpn_HRE_001",
      "parent_phage": "PKP001",
      "construct_status": "in_vitro_tested",
      "target_pathogen_text": "KL47-type Klebsiella pneumoniae",
      "intended_effects": [
        "扩宿主谱"
      ],
      "strategy_refs": [
        "tail_fiber_engineering",
        "host_range_expa

In [5]:
# ============================================================
# Cell 5: 落库（全部 pending）
# ============================================================
ids = persist_candidates(driver, extraction, actor_id="demo")
print("📦 落库结果：")
for k, v in ids.items():
    print(f"  {k}: {len(v)} 条 → {v[:3]}{'...' if len(v)>3 else ''}")

📦 落库结果：
  strategy_ids: 2 条 → ['ENG:STRAT:CE258659', 'ENG:STRAT:33F089EB']
  construct_ids: 1 条 → ['ENG:CONST:EAE7F653']
  claim_ids: 1 条 → ['ENG:CLAIM:B75855B5']
  result_ids: 3 条 → ['ENG:RESULT:5A5297AB', 'ENG:RESULT:FFFF9EAD', 'ENG:RESULT:4FEF681C']


In [6]:
from config import get_driver
driver = get_driver()
with driver.session() as s:
    s.run("""
        MATCH (ec:EngineeredPhageConstruct)
        WHERE ec.target_pathogen_text IS NULL
        SET ec.target_pathogen_text = ''
    """)
print("✅ target_pathogen_text 属性补齐完成")

✅ target_pathogen_text 属性补齐完成


In [7]:
# ============================================================
# Cell 6: 列出待审候选对象（含复用）
# ============================================================
extraction_id = src_id  # 上一次抽取用的 source
with driver.session() as s:
    print("=== 本次抽取产生的 / 复用的策略 ===")
    for r in s.run("""
        MATCH (es:EngineeringStrategy)-[:SUPPORTED_BY]->(src:SourceArtifact {source_id: $sid})
        RETURN es.strategy_id AS id, es.strategy_type AS type,
               es.review_status AS review_status,
               COALESCE(es.extracted_by, 'human') AS by
    """, sid=extraction_id):
        print(dict(r))

    print("\n=== 本次抽取产生的 / 复用的构建体 ===")
    for r in s.run("""
        MATCH (ec:EngineeredPhageConstruct)-[:REPORTED_IN]->(src:SourceArtifact {source_id: $sid})
        RETURN ec.construct_id AS id, ec.public_name AS name,
               ec.construct_status AS status,
               ec.review_status AS review_status,
               COALESCE(ec.extracted_by, 'human') AS by
    """, sid=extraction_id):
        print(dict(r))

print("\n=== 本次抽取的 claims / results（按 pending 过滤）===")
pending = list_pending_candidates(driver)
for k, v in pending.items():
    print(f"  {k}: {len(v)} 条")

=== 本次抽取产生的 / 复用的策略 ===
{'id': 'ENG:STRAT:33F089EB', 'type': 'host_range_expansion', 'review_status': 'approved', 'by': 'human'}
{'id': 'ENG:STRAT:CE258659', 'type': 'tail_fiber_engineering', 'review_status': 'approved', 'by': 'human'}

=== 本次抽取产生的 / 复用的构建体 ===
{'id': 'ENG:CONST:EAE7F653', 'name': 'vB_Kpn_HRE_001', 'status': 'in_vitro_tested', 'review_status': 'pending', 'by': 'human'}

=== 本次抽取的 claims / results（按 pending 过滤）===
  strategies: 0 条
  constructs: 0 条
  claims: 4 条
  results: 6 条


In [8]:
# ============================================================
# Cell 7: 审核通过 / 拒绝
# ============================================================
# 举例：审核通过第一个 claim
if pending["claims"]:
    cid = pending["claims"][0]["id"]
    approve_candidate(
        driver,
        candidate_type="claim",
        node_id=cid,
        reviewer_id="expert_wang",
        comment="抽取准确，与原文一致，审核通过",
    )
    print(f"✅ 已通过 claim {cid}")

if pending["claims"] and len(pending["claims"]) > 1:
    cid2 = pending["claims"][1]["id"]
    reject_candidate(
        driver,
        candidate_type="claim",
        node_id=cid2,
        reviewer_id="expert_wang",
        comment="该主张在原文中为推测性表述，证据不足，暂不采纳",
    )
    print(f"❌ 已拒绝 claim {cid2}")

✅ 已通过 claim ENG:CLAIM:B75855B5
❌ 已拒绝 claim ENG:CLAIM:CCB8F395


In [9]:
# ============================================================
# Cell 8: 验证：只有 approved 的 claim 才进入技术简报
# ============================================================
from src.engineering_intelligence.technology_brief import generate_technology_brief
from src.engineering_intelligence.strategy_classifier import get_all_strategies

strategies = get_all_strategies(driver)
if strategies:
    brief = generate_technology_brief(
        driver,
        strategy_id=strategies[0]["id"],
        persist=False,
    )
    print("技术简报包含的 Claim 数：", len(brief["technical_claims"]))
    print("技术简报包含的 Result 数：", len(brief["technical_results"]))
    print("（预期：只包含 approved 的，pending 的不会出现在这里）")

技术简报包含的 Claim 数： 2
技术简报包含的 Result 数： 0
（预期：只包含 approved 的，pending 的不会出现在这里）


In [10]:
# ============================================================
# Cell 9: 校验未审核候选对象没有进入简报
# ============================================================
pending_after = list_pending_candidates(driver)
print("剩余待审候选对象：")
for k, v in pending_after.items():
    print(f"  {k}: {len(v)} 条")

with driver.session() as s:
    leaked = s.run("""
        MATCH (tc:TechnicalClaim)-[:CLAIMS_ABOUT]->(:EngineeredPhageConstruct)
        WHERE tc.extracted_by = 'deepseek'
          AND COALESCE(tc.review_status, 'approved') <> 'approved'
        RETURN count(tc) AS c
    """).single()["c"]
    print(f"\n未审核但已建立 CLAIMS_ABOUT 关系的 claim：{leaked} 条（应 > 0，因为 pending 也会建关系，但会被 brief 过滤）")

剩余待审候选对象：
  strategies: 0 条
  constructs: 0 条
  claims: 2 条
  results: 6 条

未审核但已建立 CLAIMS_ABOUT 关系的 claim：4 条（应 > 0，因为 pending 也会建关系，但会被 brief 过滤）
